# This is my Plan (do not generete code)
My fianl goal is to create a table where each row is an user id, and I have columns such as:
1) Num like day one. 
2) Num posts day two.
3) Num of blocks day three. 
4) Joining date.

In [ ]:
# Load the filtered blocks
blocks_table = pq.read_table(blocks_output_path)
blocks_df = blocks_table.to_pandas()

print(f"Filtered blocks loaded: {len(blocks_df)}")
print("\nBlocks schema:")
print(blocks_table.schema)
print("\nFirst few rows:")
print(blocks_df.head())

Filtered blocks loaded: 265695

Blocks schema:
created_at: timestamp[us, tz=UTC]
did_id: int64
subject_id: int64
-- schema metadata --
pandas: '{"index_columns": [], "column_indexes": [], "columns": [{"name":' + 465

First few rows:
                        created_at  did_id  subject_id
0 2024-11-16 15:16:16.539000+00:00     180     3640019
1 2024-11-25 13:34:08.890000+00:00     219       51454
2 2024-11-28 08:42:33.595000+00:00     219    15318020
3 2025-01-19 22:57:17.917000+00:00     219     1664267
4 2025-01-22 22:04:05.166000+00:00     219    32226456


## Merging and Processing
We want a table where for each user we have the following stuff: 
1) Join date timestamp. 
2) First post timestamp.
3) day_1_posts, day_2_posts, ..., day_n_posts.
4) TODO: Look at the block database. 
5) TODO: Look at the likes database.

In [ ]:
# Inner join profiles with posts
merged_df = filtered_posts_df.merge(
    active_profiles, on='did_id', how='inner'
)
print(merged_df.head())

                        created_at  did_id                        join_date
0 2024-08-30 21:25:28.002000+00:00     318 2024-08-30 21:04:26.303000+00:00
1 2024-08-30 22:29:13.143000+00:00     318 2024-08-30 21:04:26.303000+00:00
2 2024-08-31 14:01:03.295000+00:00     318 2024-08-30 21:04:26.303000+00:00
3 2024-08-31 14:23:46.432000+00:00     318 2024-08-30 21:04:26.303000+00:00
4 2024-08-31 14:39:12.549000+00:00     318 2024-08-30 21:04:26.303000+00:00


In [ ]:
# Create a user_table with "did_id", "user_join_date" and "user_first_post"
user_first_post = merged_df.groupby('did_id')['created_at'].min().reset_index()
user_first_post = user_first_post.rename(columns={'created_at': 'first_post_date'})

user_table = active_profiles[['did_id', 'join_date']].merge(user_first_post, on='did_id', how='inner')
print("User Table")
print(user_table.head())

User Table
   did_id                        join_date                  first_post_date
0     318 2024-08-30 21:04:26.303000+00:00 2024-08-30 21:25:28.002000+00:00
1    1032 2024-10-19 12:08:26.894000+00:00 2024-11-17 23:04:06.860000+00:00
2    1738 2024-11-24 22:41:30.245000+00:00 2024-11-24 22:44:22.029000+00:00
3    3316 2024-06-08 03:54:37.834000+00:00 2024-06-08 06:02:30.972000+00:00
4    3360 2024-11-17 05:41:26.862000+00:00 2024-11-17 05:44:20.897000+00:00


In [ ]:
# Calculate days since joining, filter the first month of activity, merge them to user_table

merged_df['days_since_join'] = (
    (merged_df['created_at'] - merged_df['join_date']).dt.total_seconds() / (24 * 3600)
).round().astype(int)

first_month_posts = merged_df[
    (merged_df['days_since_join'] >= 0) & 
    (merged_df['days_since_join'] <= 15)
]

daily_post_counts = first_month_posts.groupby(['did_id', 'days_since_join']).size().reset_index(name='post_count')

# Pivot
time_series_wide = daily_post_counts.pivot_table(
    index='did_id', 
    columns='days_since_join', 
    values='post_count', 
    fill_value=0
).reset_index()

# Rename the day-posts columns
time_series_wide.columns = ['did_id'] + [f'day_{int(col)}_posts' for col in time_series_wide.columns[1:]]

# Merge into final table
user_table = user_table.merge(time_series_wide, on='did_id', how='left')

print(user_table.head())

   did_id                        join_date                  first_post_date  \
0     318 2024-08-30 21:04:26.303000+00:00 2024-08-30 21:25:28.002000+00:00   
1    1032 2024-10-19 12:08:26.894000+00:00 2024-11-17 23:04:06.860000+00:00   
2    1738 2024-11-24 22:41:30.245000+00:00 2024-11-24 22:44:22.029000+00:00   
3    3316 2024-06-08 03:54:37.834000+00:00 2024-06-08 06:02:30.972000+00:00   
4    3360 2024-11-17 05:41:26.862000+00:00 2024-11-17 05:44:20.897000+00:00   

   day_0_posts  day_1_posts  day_2_posts  day_3_posts  day_4_posts  \
0          2.0          5.0          4.0          4.0          3.0   
1          0.0          0.0          0.0          0.0          0.0   
2         19.0          0.0          0.0          0.0          0.0   
3          5.0          0.0          0.0          1.0          0.0   
4          1.0         18.0         11.0          4.0         20.0   

   day_5_posts  day_6_posts  ...  day_21_posts  day_22_posts  day_23_posts  \
0          3.0          0.

# Saving 

In [ ]:
# Save the processed data for future use
output_path = "../data/posting/processed/user_activity.parquet"
user_table.to_parquet(output_path, index=False)
print(f"\nData saved to: {output_path}")


Data saved to: ../data/posting/processed/user_activity.parquet
